# Monte Carlo Price Perturbation Simulation

The bankroll simulation is fully deterministic — same movies, same prices, same result.
With-replacement bootstrapping tests composition risk but introduces duplication artifacts.

**Idea:** Perturb market prices with empirical noise. The model side (reviews, KDE, p_fresh)
stays fixed — only the market price changes, which shifts when edge first crosses min_edge
and at what entry price. Since `edge = model_p_yes * 100 - market_price`, no KDE recomputation needed.

**Plan:** `plans/plan_monte_carlo_price_perturbation.md`

In [ ]:
import sys, os, io, glob, contextlib, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from edge import compute_edge
from critic_model import (
    build_critic_profiles, build_kde_lambda_model,
    default_training_slugs, estimate_lambda, estimate_p_fresh,
)

PRICE_DIR = ROOT / "rt-price-histories"
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
reviews_df = pd.read_csv(ROOT / "reviews.csv")
reviews_df["estimated_timestamp"] = pd.to_datetime(
    reviews_df["estimated_timestamp"], format="ISO8601", utc=True
)
movies_df = pd.read_csv(ROOT / "movies_index.csv")
movies_df["Bet Close Date"] = pd.to_datetime(movies_df["Bet Close Date"], utc=True)

slugs_with_prices = sorted([
    d.name for d in PRICE_DIR.iterdir()
    if d.is_dir() and list(d.glob("*hour*"))
])
movies_bt = movies_df[movies_df["Slug"].isin(slugs_with_prices)].copy()
movies_bt = movies_bt.dropna(subset=["Bet Close Date"]).sort_values("Bet Close Date")
print(f"Reviews: {len(reviews_df):,} rows, {reviews_df['movie_slug'].nunique()} movies")
print(f"Movies with price data: {len(movies_bt)}")

## Step 1: Characterize price noise

Measure the empirical distribution of hour-to-hour price changes across all movies and thresholds.
Split by time-to-close bucket to see if volatility differs.

In [ ]:
def load_hourly_prices(slug):
    csv_files = list((PRICE_DIR / slug).glob("*hour*"))
    if not csv_files:
        return None
    df = pd.read_csv(csv_files[0])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)
    thresh_cols = [c for c in df.columns if c.startswith("Above ")]
    df[thresh_cols] = df[thresh_cols].ffill()
    return df

def get_resolution(price_df):
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    resolution = {}
    for col in thresh_cols:
        thresh = int(col.split()[-1])
        last_valid = price_df[col].dropna()
        if last_valid.empty:
            resolution[thresh] = None
            continue
        terminal = last_valid.iloc[-1]
        if terminal >= 90:
            resolution[thresh] = True
        elif terminal <= 10:
            resolution[thresh] = False
        else:
            resolution[thresh] = None
    return resolution

# Collect all hour-to-hour deltas across all movies
all_deltas = []

for slug in movies_bt["Slug"]:
    pdf = load_hourly_prices(slug)
    if pdf is None or len(pdf) < 2:
        continue
    market_close = pdf["timestamp"].iloc[-1]
    thresh_cols = [c for c in pdf.columns if c.startswith("Above ")]
    
    for col in thresh_cols:
        series = pdf[col].dropna()
        if len(series) < 2:
            continue
        prices = series.values
        timestamps = pdf.loc[series.index, "timestamp"]
        hours_to_close = (market_close - timestamps).dt.total_seconds() / 3600
        
        deltas = np.diff(prices)
        htc = hours_to_close.values[:-1]  # hours_to_close at the start of each interval
        price_levels = prices[:-1]
        
        for d, h, p in zip(deltas, htc, price_levels):
            if np.isfinite(d) and np.isfinite(h):
                all_deltas.append({
                    "delta": d,
                    "hours_to_close": h,
                    "price_level": p,
                    "slug": slug,
                })

deltas_df = pd.DataFrame(all_deltas)
print(f"Collected {len(deltas_df):,} hour-to-hour price deltas across {deltas_df['slug'].nunique()} movies")

In [ ]:
# Overall delta statistics
print("=== Overall hour-to-hour delta statistics ===")
print(f"Mean:   {deltas_df['delta'].mean():.3f}c")
print(f"Median: {deltas_df['delta'].median():.3f}c")
print(f"Std:    {deltas_df['delta'].std():.3f}c")
print(f"Skew:   {deltas_df['delta'].skew():.3f}")
print(f"Kurt:   {deltas_df['delta'].kurtosis():.3f}")
print()

# By time-to-close bucket (action window is 24-120h)
buckets = [
    ("T-7d+", 168, 9999),
    ("T-5d to T-3d", 72, 120),
    ("T-3d to T-1d", 24, 72),
    ("< T-1d", 0, 24),
]
print("=== Delta std by time-to-close bucket ===")
for label, lo, hi in buckets:
    mask = (deltas_df["hours_to_close"] >= lo) & (deltas_df["hours_to_close"] < hi)
    sub = deltas_df.loc[mask, "delta"]
    if len(sub) > 10:
        print(f"  {label:20s}: std={sub.std():.3f}c, mean={sub.mean():.3f}c, n={len(sub):,}")

print()

# By price level bucket
print("=== Delta std by price level ===")
for lo, hi in [(1, 20), (20, 40), (40, 60), (60, 80), (80, 99)]:
    mask = (deltas_df["price_level"] >= lo) & (deltas_df["price_level"] < hi)
    sub = deltas_df.loc[mask, "delta"]
    if len(sub) > 10:
        print(f"  price [{lo:2d}-{hi:2d}): std={sub.std():.3f}c, mean={sub.mean():.3f}c, n={len(sub):,}")

In [ ]:
# Visualize delta distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Overall histogram
ax = axes[0]
clip = deltas_df["delta"].clip(-20, 20)
ax.hist(clip, bins=80, density=True, alpha=0.7, edgecolor="none")
ax.axvline(0, color="k", linewidth=0.5)
ax.set_xlabel("Hour-to-hour price delta (cents)")
ax.set_ylabel("Density")
ax.set_title(f"All deltas (std={deltas_df['delta'].std():.2f}c)")

# QQ plot vs normal
ax = axes[1]
sample = deltas_df["delta"].dropna().values
sample = sample[np.isfinite(sample)]
sp_stats.probplot(sample[::max(1, len(sample)//5000)], dist="norm", plot=ax)
ax.set_title("QQ plot vs Normal")

# Std by time-to-close
ax = axes[2]
htc_bins = np.arange(0, 250, 12)
stds = []
centers = []
for i in range(len(htc_bins) - 1):
    mask = (deltas_df["hours_to_close"] >= htc_bins[i]) & (deltas_df["hours_to_close"] < htc_bins[i+1])
    sub = deltas_df.loc[mask, "delta"]
    if len(sub) > 20:
        stds.append(sub.std())
        centers.append((htc_bins[i] + htc_bins[i+1]) / 2)
ax.bar(centers, stds, width=10, alpha=0.7)
ax.set_xlabel("Hours to close")
ax.set_ylabel("Delta std (cents)")
ax.set_title("Volatility by time-to-close")
ax.axvspan(24, 120, alpha=0.1, color="green", label="Action window")
ax.legend()

plt.tight_layout()
plt.show()

## Step 2: Backtest with cached model_p_yes

Run the full backtest once (daily snapshots), caching `model_p_yes` at each snapshot.
The MC engine will reuse these: `perturbed_edge = model_p_yes * 100 - perturbed_price`.

In [ ]:
def precompute_review_states(slug, reviews_df, bet_close):
    movie_reviews = reviews_df[reviews_df["movie_slug"] == slug].copy()
    movie_reviews = movie_reviews[movie_reviews["estimated_timestamp"] <= bet_close]
    movie_reviews = movie_reviews.sort_values("estimated_timestamp").reset_index(drop=True)
    if movie_reviews.empty:
        return [], None
    states = []
    critics = set()
    fresh = 0
    total = 0
    for _, row in movie_reviews.iterrows():
        critics = critics | {row["reviewer_name"]}
        total += 1
        if row["tomatometer_sentiment"] == "positive":
            fresh += 1
        states.append({
            "timestamp": row["estimated_timestamp"],
            "observed_critics": frozenset(critics),
            "fresh_count": fresh,
            "total_count": total,
        })
    return states, movie_reviews["estimated_timestamp"].iloc[0]


def get_review_state_at(states, snapshot_time):
    if not states or snapshot_time < states[0]["timestamp"]:
        return set(), 0, 0
    best = None
    for s in states:
        if s["timestamp"] <= snapshot_time:
            best = s
        else:
            break
    if best is None:
        return set(), 0, 0
    return best["observed_critics"], best["fresh_count"], best["total_count"]

In [ ]:
def backtest_movie_cached(slug, reviews_df, movies_df, every_n_hours=24):
    """Run backtest for one movie, returning snapshot-level records with cached model_p_yes."""
    row = movies_df[movies_df["Slug"] == slug].iloc[0]
    bet_close_date = row["Bet Close Date"]
    price_df = load_hourly_prices(slug)
    if price_df is None or price_df.empty:
        return []
    market_close_time = price_df["timestamp"].iloc[-1]
    resolution = get_resolution(price_df)
    training_slugs = default_training_slugs(
        movies_df, exclude_slug=slug, before_date=bet_close_date
    )
    if len(training_slugs) < 5:
        return []
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        profiles = build_critic_profiles(reviews_df, movies_df, training_slugs)
        model = build_kde_lambda_model(profiles)
    review_states, first_review_ts = precompute_review_states(slug, reviews_df, market_close_time)
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    records = []
    cached_critics, cached_fresh, cached_total = set(), 0, 0
    cached_lambda, cached_p_fresh = None, None
    last_kept_ts = None

    for i in range(len(price_df)):
        snap_row = price_df.iloc[i]
        snapshot_time = snap_row["timestamp"]
        if every_n_hours > 1 and last_kept_ts is not None:
            hours_since = (snapshot_time - last_kept_ts).total_seconds() / 3600
            if hours_since < every_n_hours:
                continue
        last_kept_ts = snapshot_time
        hours_to_close = (market_close_time - snapshot_time).total_seconds() / 3600
        if hours_to_close <= 0:
            continue
        days_before_close = hours_to_close / 24
        observed_critics, fresh_count, total_count = get_review_state_at(
            review_states, snapshot_time
        )
        state_changed = (total_count != cached_total)
        if state_changed or cached_lambda is None:
            cached_critics = observed_critics
            cached_fresh = fresh_count
            cached_total = total_count
            first_review_dbc = None
            if first_review_ts is not None:
                first_review_dbc = (market_close_time - first_review_ts).total_seconds() / 86400
            with contextlib.redirect_stdout(io.StringIO()):
                cached_lambda = estimate_lambda(
                    model, days_before_close, hours_to_close,
                    observed_critics, observed_count=total_count,
                    first_review_dbc=first_review_dbc,
                )
                cached_p_fresh = estimate_p_fresh(
                    profiles, observed_critics, fresh_count, total_count,
                )
        if cached_total == 0:
            continue

        current_score = round(cached_fresh / cached_total * 100)

        for col in thresh_cols:
            thresh = int(col.split()[-1])
            market_price = snap_row[col]
            if pd.isna(market_price) or market_price <= 0 or market_price >= 100:
                continue
            resolved = resolution.get(thresh)
            if resolved is None:
                continue
            result = compute_edge(
                threshold=thresh, market_price=market_price,
                fresh_count=cached_fresh, total_count=cached_total,
                hours_to_close=hours_to_close,
                lambda_rate=cached_lambda, p_fresh=cached_p_fresh,
            )
            records.append({
                "slug": slug,
                "threshold": thresh,
                "snapshot_time": snapshot_time,
                "hours_to_close": hours_to_close,
                "market_price": market_price,
                "model_p_yes": result["p_yes"],  # cache this for MC
                "edge_cents": result["edge_cents"],
                "current_score": current_score,
                "score_margin": current_score - thresh,
                "resolved_yes": resolved,
            })
    return records

In [ ]:
EVERY_N_HOURS = 24
all_records = []
slugs = movies_bt["Slug"].tolist()
skipped = []
t0 = time.time()

for idx, slug in enumerate(slugs):
    elapsed = time.time() - t0
    rate = (idx / elapsed) if elapsed > 0 and idx > 0 else 0
    eta = (len(slugs) - idx) / rate if rate > 0 else 0
    print(f"\r[{idx+1}/{len(slugs)}] {slug:<40s} ({elapsed:.0f}s, ~{eta:.0f}s left)", end="", flush=True)
    try:
        records = backtest_movie_cached(slug, reviews_df, movies_df, every_n_hours=EVERY_N_HOURS)
        all_records.extend(records)
    except Exception as e:
        skipped.append((slug, str(e)))

print(f"\nDone in {time.time()-t0:.0f}s. {len(all_records):,} evaluations, {len(slugs)-len(skipped)} movies.")
if skipped:
    print(f"Skipped {len(skipped)}: {[s[0] for s in skipped[:5]]}")

In [ ]:
trades = pd.DataFrame(all_records)
trades["direction"] = np.where(trades["edge_cents"] >= 0, "Yes", "No")
trades["abs_edge"] = trades["edge_cents"].abs()
trades["pnl"] = np.where(
    trades["direction"] == "Yes",
    np.where(trades["resolved_yes"], 100 - trades["market_price"], -trades["market_price"]),
    np.where(trades["resolved_yes"], -(100 - trades["market_price"]), trades["market_price"]),
)
print(f"{len(trades):,} evaluations, {trades['slug'].nunique()} movies")
print(f"model_p_yes range: [{trades['model_p_yes'].min():.4f}, {trades['model_p_yes'].max():.4f}]")

## Step 3: Monte Carlo engine

For each simulation:
1. Perturb each snapshot's market price by adding noise drawn from the empirical delta distribution.
2. Recompute edge: `perturbed_edge = model_p_yes * 100 - perturbed_price` (no KDE recomputation).
3. Apply strategy filters (No-only, min_edge, score margin band, action window).
4. Take first entry per (movie, threshold), compound bankroll.

In [ ]:
# Build the empirical noise pool: hour-to-hour deltas within the action window (24-120h)
action_window_mask = (
    (deltas_df["hours_to_close"] >= 24) & (deltas_df["hours_to_close"] <= 120)
)
noise_pool = deltas_df.loc[action_window_mask, "delta"].values
print(f"Noise pool: {len(noise_pool):,} deltas from action window [24h, 120h]")
print(f"  mean={noise_pool.mean():.3f}c, std={noise_pool.std():.3f}c")
print(f"  p5={np.percentile(noise_pool, 5):.2f}c, p95={np.percentile(noise_pool, 95):.2f}c")

In [ ]:
def mc_simulate(trades_df, noise_pool, n_sims, min_edge, bankroll_frac,
                margin_floor=None, margin_ceil=None,
                start_bankroll=100000.0, noise_scale=1.0,
                rng_seed=42):
    """Run Monte Carlo price perturbation simulation.
    
    For each sim, perturb every snapshot's market price by drawing from noise_pool,
    then re-evaluate strategy using cached model_p_yes.
    
    Returns array of final bankroll multipliers (length n_sims).
    Also returns per-movie details for the last sim.
    """
    ACTION_WINDOW = (24, 120)
    rng = np.random.default_rng(rng_seed)
    
    # Pre-filter to action window only (saves work inside the loop)
    window_mask = (
        (trades_df["hours_to_close"] >= ACTION_WINDOW[0]) &
        (trades_df["hours_to_close"] <= ACTION_WINDOW[1])
    )
    df = trades_df[window_mask].copy()
    
    # Vectorized columns we'll need
    model_p_yes = df["model_p_yes"].values
    actual_prices = df["market_price"].values
    score_margins = df["score_margin"].values
    resolved_yes = df["resolved_yes"].values
    slugs = df["slug"].values
    thresholds = df["threshold"].values
    snapshot_times = df["snapshot_time"].values
    
    n_rows = len(df)
    multipliers = np.empty(n_sims)
    
    for sim in range(n_sims):
        # Perturb prices
        noise = rng.choice(noise_pool, size=n_rows) * noise_scale
        perturbed_prices = np.clip(actual_prices + noise, 1, 99)
        
        # Recompute edge with perturbed prices
        perturbed_edge = model_p_yes * 100 - perturbed_prices
        
        # Apply strategy filters: No-only
        direction_no = perturbed_edge < 0
        abs_edge = np.abs(perturbed_edge)
        edge_ok = abs_edge >= min_edge
        
        margin_ok = np.ones(n_rows, dtype=bool)
        if margin_floor is not None:
            margin_ok &= score_margins >= margin_floor
        if margin_ceil is not None:
            margin_ok &= score_margins <= margin_ceil
        
        mask = direction_no & edge_ok & margin_ok
        
        if not mask.any():
            multipliers[sim] = 1.0
            continue
        
        # Get filtered signals
        idx = np.where(mask)[0]
        # Sort by snapshot time to get first entry per (slug, threshold)
        sort_order = np.argsort(snapshot_times[idx])
        idx_sorted = idx[sort_order]
        
        # First entry per (slug, threshold)
        seen = set()
        positions = []
        for i in idx_sorted:
            key = (slugs[i], thresholds[i])
            if key not in seen:
                seen.add(key)
                # No-side P&L: if resolved No (not Yes), we win the entry cost
                entry_cost = 100 - perturbed_prices[i]
                if not resolved_yes[i]:
                    pos_pnl = perturbed_prices[i]  # win: collect the No price
                else:
                    pos_pnl = -entry_cost  # lose: lose the entry cost
                positions.append((slugs[i], pos_pnl, entry_cost, snapshot_times[i]))
        
        if not positions:
            multipliers[sim] = 1.0
            continue
        
        # Aggregate to per-movie P&L, compound bankroll
        pos_df = pd.DataFrame(positions, columns=["slug", "pos_pnl", "entry_cost", "time"])
        movie_agg = pos_df.groupby("slug").agg(
            total_pnl=("pos_pnl", "sum"),
            total_cost=("entry_cost", "sum"),
            first_time=("time", "min"),
        ).sort_values("first_time")
        
        bankroll = start_bankroll
        for _, mr in movie_agg.iterrows():
            risk_budget = bankroll * bankroll_frac
            scale = risk_budget / mr["total_cost"] if mr["total_cost"] > 0 else 0
            bankroll += scale * mr["total_pnl"]
            if bankroll <= 0:
                bankroll = 0
                break
        
        multipliers[sim] = bankroll / start_bankroll
    
    return multipliers

## Sanity check: zero noise should reproduce ACTUAL

With `noise_scale=0`, the MC should return the same multiplier as the deterministic simulation.

In [ ]:
# Zero-noise sanity check
zero_mult = mc_simulate(
    trades, noise_pool, n_sims=1, min_edge=20, bankroll_frac=0.10,
    margin_floor=-3, margin_ceil=3, noise_scale=0.0,
)
print(f"Zero-noise MC multiplier: {zero_mult[0]:.1f}x")
print(f"Expected (ACTUAL from margin_bankroll_sim): ~249.0x")
print(f"Match: {'YES' if abs(zero_mult[0] - 249.0) < 5 else 'NO — investigate!'}")

## Step 4: Small validation run (100 sims)

In [ ]:
t0 = time.time()
mults_100 = mc_simulate(
    trades, noise_pool, n_sims=100, min_edge=20, bankroll_frac=0.10,
    margin_floor=-3, margin_ceil=3, noise_scale=1.0,
)
elapsed = time.time() - t0
print(f"100 sims in {elapsed:.1f}s ({elapsed/100*1000:.0f}ms/sim)")
print(f"Median: {np.median(mults_100):.1f}x")
print(f"Mean:   {np.mean(mults_100):.1f}x")
print(f"Std:    {np.std(mults_100):.1f}x")
print(f"Min:    {np.min(mults_100):.1f}x")
print(f"Max:    {np.max(mults_100):.1f}x")
print(f"p5:     {np.percentile(mults_100, 5):.1f}x")
print(f"p95:    {np.percentile(mults_100, 95):.1f}x")
print(f"ACTUAL (249x) percentile: {(mults_100 <= 249).mean()*100:.0f}%")

## Step 5: Full simulation (5,000 sims) — multiple configs

In [ ]:
N_SIMS = 5000

configs = [
    {"label": "20c, [-3,+3]",  "min_edge": 20, "margin_floor": -3, "margin_ceil": 3},
    {"label": "20c, no filter", "min_edge": 20, "margin_floor": None, "margin_ceil": None},
    {"label": "15c, [-3,+3]",  "min_edge": 15, "margin_floor": -3, "margin_ceil": 3},
    {"label": "15c, no filter", "min_edge": 15, "margin_floor": None, "margin_ceil": None},
    {"label": "10c, [-3,+3]",  "min_edge": 10, "margin_floor": -3, "margin_ceil": 3},
    {"label": "10c, no filter", "min_edge": 10, "margin_floor": None, "margin_ceil": None},
]

mc_results = {}
t0 = time.time()

for cfg in configs:
    label = cfg["label"]
    print(f"Running {label}...", end=" ", flush=True)
    t1 = time.time()
    mults = mc_simulate(
        trades, noise_pool, n_sims=N_SIMS, min_edge=cfg["min_edge"],
        bankroll_frac=0.10,
        margin_floor=cfg["margin_floor"], margin_ceil=cfg["margin_ceil"],
        noise_scale=1.0, rng_seed=42,
    )
    mc_results[label] = mults
    print(f"{time.time()-t1:.0f}s, median={np.median(mults):.1f}x")

print(f"\nTotal: {time.time()-t0:.0f}s")

In [ ]:
# Summary table
# ACTUAL values from margin_bankroll_sim.ipynb / findings/score_margin_and_robustness.md
actual_values = {
    "20c, [-3,+3]": 249.0,
    "20c, no filter": 147.1,
    "15c, [-3,+3]": 192.7,
    "15c, no filter": 141.2,
    "10c, [-3,+3]": 188.4,
    "10c, no filter": 94.3,
}

rows = []
for label, mults in mc_results.items():
    actual = actual_values.get(label, None)
    pctl = np.sum(mults <= actual) / len(mults) * 100 if actual else None
    rows.append({
        "config": label,
        "ACTUAL": f"{actual:.0f}x" if actual else "?",
        "mc_p1": f"{np.percentile(mults, 1):.0f}x",
        "mc_p5": f"{np.percentile(mults, 5):.0f}x",
        "mc_p25": f"{np.percentile(mults, 25):.0f}x",
        "mc_median": f"{np.median(mults):.0f}x",
        "mc_p75": f"{np.percentile(mults, 75):.0f}x",
        "mc_p95": f"{np.percentile(mults, 95):.0f}x",
        "mc_std": f"{np.std(mults):.0f}x",
        "ACTUAL_pctl": f"{pctl:.0f}%" if pctl is not None else "?",
    })

summary_df = pd.DataFrame(rows)
print(f"=== Monte Carlo Price Perturbation: {N_SIMS} sims, 1x empirical noise ===")
print(summary_df.to_string(index=False))

In [ ]:
# Histograms of multiplier distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, (label, mults) in zip(axes.flat, mc_results.items()):
    actual = actual_values.get(label)
    ax.hist(mults, bins=60, density=True, alpha=0.7, edgecolor="none")
    if actual:
        ax.axvline(actual, color="red", linewidth=1.5, linestyle="--", label=f"ACTUAL={actual:.0f}x")
    ax.axvline(np.median(mults), color="orange", linewidth=1.5, label=f"median={np.median(mults):.0f}x")
    ax.set_xlabel("Bankroll multiplier")
    ax.set_ylabel("Density")
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle(f"Monte Carlo Price Perturbation ({N_SIMS} sims)", fontsize=13)
plt.tight_layout()
plt.show()

## Noise sensitivity: 0.5x, 1x, 2x empirical std

How much do results change when we vary the noise magnitude?

In [ ]:
# Test with the primary config (20c, [-3,+3]) at different noise scales
noise_scales = [0.5, 1.0, 2.0]
sensitivity_results = {}

for scale in noise_scales:
    print(f"noise_scale={scale}x...", end=" ", flush=True)
    t1 = time.time()
    mults = mc_simulate(
        trades, noise_pool, n_sims=N_SIMS, min_edge=20, bankroll_frac=0.10,
        margin_floor=-3, margin_ceil=3, noise_scale=scale, rng_seed=42,
    )
    sensitivity_results[scale] = mults
    print(f"{time.time()-t1:.0f}s")

print("\n=== Noise sensitivity: 20c, [-3,+3] ===")
for scale, mults in sensitivity_results.items():
    actual = 249.0
    pctl = np.sum(mults <= actual) / len(mults) * 100
    print(f"  {scale}x: median={np.median(mults):.0f}x, p5={np.percentile(mults, 5):.0f}x, "
          f"p95={np.percentile(mults, 95):.0f}x, std={np.std(mults):.0f}x, "
          f"ACTUAL@{pctl:.0f}%ile")

In [ ]:
# Overlay histograms for noise sensitivity
fig, ax = plt.subplots(figsize=(12, 5))
colors = {0.5: "blue", 1.0: "green", 2.0: "red"}

for scale, mults in sensitivity_results.items():
    ax.hist(mults, bins=60, density=True, alpha=0.3, color=colors[scale],
            label=f"{scale}x noise (med={np.median(mults):.0f}x)")

ax.axvline(249.0, color="black", linewidth=1.5, linestyle="--", label="ACTUAL=249x")
ax.set_xlabel("Bankroll multiplier")
ax.set_ylabel("Density")
ax.set_title("Noise sensitivity: 20c, [-3,+3]")
ax.legend()
plt.tight_layout()
plt.show()

## Per-movie entry sensitivity

Which movies have the most variation in entry decisions across MC runs?
A movie where edge is barely above min_edge will flip in/out of the signal set with small price perturbations.

In [ ]:
def mc_per_movie_sensitivity(trades_df, noise_pool, n_sims, min_edge, bankroll_frac,
                              margin_floor=None, margin_ceil=None, rng_seed=42):
    """Track which movies get entered in each MC sim to measure entry stability."""
    ACTION_WINDOW = (24, 120)
    rng = np.random.default_rng(rng_seed)
    
    window_mask = (
        (trades_df["hours_to_close"] >= ACTION_WINDOW[0]) &
        (trades_df["hours_to_close"] <= ACTION_WINDOW[1])
    )
    df = trades_df[window_mask].copy()
    
    model_p_yes = df["model_p_yes"].values
    actual_prices = df["market_price"].values
    score_margins = df["score_margin"].values
    resolved_yes_arr = df["resolved_yes"].values
    slugs = df["slug"].values
    thresholds = df["threshold"].values
    snapshot_times = df["snapshot_time"].values
    
    n_rows = len(df)
    
    # Track: for each movie, how many sims it appears in
    all_movies = set(slugs)
    movie_entry_count = {m: 0 for m in all_movies}
    movie_entry_prices = {m: [] for m in all_movies}
    
    for sim in range(n_sims):
        noise = rng.choice(noise_pool, size=n_rows)
        perturbed_prices = np.clip(actual_prices + noise, 1, 99)
        perturbed_edge = model_p_yes * 100 - perturbed_prices
        
        direction_no = perturbed_edge < 0
        abs_edge = np.abs(perturbed_edge)
        edge_ok = abs_edge >= min_edge
        
        margin_ok = np.ones(n_rows, dtype=bool)
        if margin_floor is not None:
            margin_ok &= score_margins >= margin_floor
        if margin_ceil is not None:
            margin_ok &= score_margins <= margin_ceil
        
        mask = direction_no & edge_ok & margin_ok
        idx = np.where(mask)[0]
        sort_order = np.argsort(snapshot_times[idx])
        idx_sorted = idx[sort_order]
        
        seen_movies = set()
        for i in idx_sorted:
            slug = slugs[i]
            if slug not in seen_movies:
                seen_movies.add(slug)
                movie_entry_count[slug] = movie_entry_count.get(slug, 0) + 1
                movie_entry_prices.setdefault(slug, []).append(perturbed_prices[i])
    
    return movie_entry_count, movie_entry_prices

entry_count, entry_prices = mc_per_movie_sensitivity(
    trades, noise_pool, n_sims=1000, min_edge=20, bankroll_frac=0.10,
    margin_floor=-3, margin_ceil=3,
)

# Movies that appear in <100% but >0% of sims — these are the sensitive ones
entry_rates = {m: c / 1000 for m, c in entry_count.items() if c > 0}
sensitive = {m: r for m, r in entry_rates.items() if 0.05 < r < 0.95}
stable = {m: r for m, r in entry_rates.items() if r >= 0.95}
rare = {m: r for m, r in entry_rates.items() if r <= 0.05}

print(f"Movies entered in 95-100% of sims (stable):  {len(stable)}")
print(f"Movies entered in 5-95% of sims (sensitive): {len(sensitive)}")
print(f"Movies entered in 0-5% of sims (rare):       {len(rare)}")
print(f"Movies never entered:                         {sum(1 for c in entry_count.values() if c == 0)}")

if sensitive:
    print(f"\nTop 10 most sensitive movies (entry rate):")
    for m, r in sorted(sensitive.items(), key=lambda x: abs(x[1] - 0.5))[:10]:
        prices = entry_prices[m]
        print(f"  {m:<40s} entry_rate={r:.1%}, price_std={np.std(prices):.1f}c")

## Comparison: MC price perturbation vs with-replacement bootstrap

The bootstrap tests composition risk (different movie mixes). The MC tests price risk (same movies, different entry prices). These are complementary sources of uncertainty.

In [ ]:
# Bootstrap results from findings/score_margin_and_robustness.md
bootstrap_data = {
    "20c, [-3,+3]": {"repl_p5": 31.3, "repl_med": 243.9, "repl_p95": 2094.0, "repl_std": 1605},
    "20c, no filter": {"repl_p5": 26.4, "repl_med": 138.5, "repl_p95": 919.7, "repl_std": 467},
    "15c, [-3,+3]": {"repl_p5": 29.4, "repl_med": 186.9, "repl_p95": 1453.9, "repl_std": 844},
    "15c, no filter": {"repl_p5": 29.7, "repl_med": 131.8, "repl_p95": 793.4, "repl_std": 403},
    "10c, [-3,+3]": {"repl_p5": 28.0, "repl_med": 182.4, "repl_p95": 1405.7, "repl_std": 935},
    "10c, no filter": {"repl_p5": 22.7, "repl_med": 92.1, "repl_p95": 458.8, "repl_std": 247},
}

comp_rows = []
for label in mc_results:
    mults = mc_results[label]
    bs = bootstrap_data.get(label, {})
    comp_rows.append({
        "config": label,
        "ACTUAL": actual_values.get(label),
        "mc_p5": np.percentile(mults, 5),
        "mc_med": np.median(mults),
        "mc_p95": np.percentile(mults, 95),
        "mc_std": np.std(mults),
        "bs_p5": bs.get("repl_p5"),
        "bs_med": bs.get("repl_med"),
        "bs_p95": bs.get("repl_p95"),
        "bs_std": bs.get("repl_std"),
    })

comp_df = pd.DataFrame(comp_rows)
print("=== MC price perturbation vs with-replacement bootstrap ===")
print("(MC = price risk, BS = composition risk)")
print()
for _, r in comp_df.iterrows():
    print(f"{r['config']}:")
    print(f"  ACTUAL:         {r['ACTUAL']:.0f}x")
    print(f"  MC  [p5/med/p95/std]: {r['mc_p5']:.0f}x / {r['mc_med']:.0f}x / {r['mc_p95']:.0f}x / {r['mc_std']:.0f}x")
    print(f"  BS  [p5/med/p95/std]: {r['bs_p5']:.0f}x / {r['bs_med']:.0f}x / {r['bs_p95']:.0f}x / {r['bs_std']:.0f}x")
    print(f"  MC spread (p95/p5): {r['mc_p95']/r['mc_p5']:.1f}x   BS spread: {r['bs_p95']/r['bs_p5']:.1f}x")
    print()